# Fine-tune an 8B model on 4 GB of VRAM — run it yourself

*(Open in Colab link removed along with the GitHub repo link.)*

Kadhi trains a model whose weights do not fit
in your GPU. The frozen base stays in host RAM and is streamed to the GPU one decoder
layer at a time, so peak VRAM is bounded by **one layer** instead of by the model.

This notebook does not ask you to believe that. It caps this process to **4 GB** on
Colab's free T4 and then measures what actually happens.

| Section | What it proves | Time |
|---|---|---|
| 1–3 | The cap is real, and this GPU has no bf16 | ~2 min |
| 4 | A streamed model and a normal one produce **bit-identical** logits | ~3 min |
| 5 | Llama-3.1-8B trains with a measured peak under 4 GB | ~20 min |

Sections 1–4 are the argument. Section 5 is the headline and is optional.

**Runtime → Change runtime type → T4 GPU** before you start.


## 1. Install

This notebook is pinned to **kadhi-cli==0.75.0**, the release this run was verified
against. **v0.74.0** was the first release carrying both the T4 precision fix
(#385,
#387) and the T4 GradScaler fix
(#429) that section 5 needs to
train at all on this card; 0.75.0 is pinned here only because it is the exact release
this run's committed outputs were produced against. Bump the pin only after re-running
this notebook end to end on a real T4 and recording the new outputs.

The `torchao` line is not incidental either. Colab preinstalls **torchao 0.10.0**, and
`peft` does not merely decline to use a version it considers too old — it *raises*
`ImportError` from `is_torchao_available()`, several frames inside `get_peft_model`.
Nothing here needs torchao, so it is removed rather than upgraded (upgrading risks
pulling a wheel built against a different torch).

**Colab's default Python can be newer than this project supports.** `kadhi-cli` caps at `<3.13` on purpose (untested torch/bitsandbytes wheels on newer interpreters fail with an opaque loader crash, not a clean error). Colab's hosted runtime does not let you pick an arbitrary kernel the way local Jupyter does, so if you land on Python 3.13+, the next cell falls back to installing with `--ignore-requires-python` — that trades the guarantee for a best-effort install, on the assumption that the `[train]` extras (torch,
transformers, peft, bitsandbytes, accelerate) have caught up with a 3.13 wheel by the time you are reading this. If anything below fails with an import or ABI error, that assumption did not hold on this Colab image; open an issue with the
traceback rather than treating it as this notebook's bug.


In [1]:
import sys

PY_OK = sys.version_info < (3, 13)
print(f"Colab Python: {sys.version.split()[0]}  (kadhi-cli supports <3.13)")
if not PY_OK:
    print("Falling back to --ignore-requires-python for the install below;")
    print("see the note above this cell for what that trades away.")


Colab Python: 3.13.15  (kadhi-cli supports <3.13)
Falling back to --ignore-requires-python for the install below;
see the note above this cell for what that trades away.


In [2]:
%pip uninstall -q -y torchao
import sys

_extra_flag = "--ignore-requires-python" if sys.version_info >= (3, 13) else ""
%pip install -q "kadhi-cli[train]==0.75.0" {_extra_flag}

import importlib.util

import kadhi_cli
import kadhi_cli.utils.gpu

print("kadhi", kadhi_cli.__version__)
print("pre-Ampere fix present:", hasattr(kadhi_cli.utils.gpu, "bf16_fp16_flags"))
print("torchao gone (peft raises on an old one):",
      importlib.util.find_spec("torchao") is None)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.8/475.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.3 MB/s eta 0:00:00
kadhi 0.75.0
pre-Ampere fix present: True
torchao gone (peft raises on an old one): True


In [3]:
import bitsandbytes
import peft
import torch
import transformers

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("peft        ", peft.__version__)
print("bitsandbytes", bitsandbytes.__version__)
print("kadhi_cli    ", kadhi_cli.__version__)


torch        2.11.0+cu128
transformers 5.16.1
peft         0.20.0
bitsandbytes 0.50.2
kadhi_cli     0.75.0


## 2. What card did we get, and does it have bf16?

Colab's free tier is a **T4** — Turing, sm_75. bf16 hardware arrived with Ampere, so a
T4 has none.

**Read the two lines below carefully, because they disagree, and the disagreement is
the point.** `torch.cuda.is_bf16_supported()` defaults to `including_emulation=True`:
when the compute-capability check fails it falls through to *constructing* a bf16
tensor, which software emulation satisfies. So a T4 answers **True** to the question
everyone asks, and False only to `is_bf16_supported(including_emulation=False)`.

Kadhi asked the permissive question and therefore handed bf16 to a card with no bf16
units. The first version of this fix asked it too, and was a no-op on exactly the
hardware it was written for — caught by running this notebook on a real T4, not before
(#385,
#387).


In [4]:
import torch

from kadhi_cli.utils.gpu import bf16_fp16_flags

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU."

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
total = torch.cuda.get_device_properties(0).total_memory

print(f"GPU                  {name}  (sm_{major}{minor})")
print(f"VRAM                 {total / 1e9:.1f} GB")
print(f"bf16, incl. emulation {torch.cuda.is_bf16_supported()}")
print(f"bf16 IN HARDWARE      "
      f"{torch.cuda.is_bf16_supported(including_emulation=False)}")

bf16, fp16 = bf16_fp16_flags("cuda")
print(f"Kadhi will train in   {'bf16' if bf16 else 'fp16' if fp16 else 'fp32'}")


GPU                  Tesla T4  (sm_75)
VRAM                 15.6 GB
bf16, incl. emulation True
bf16 IN HARDWARE      False
Kadhi will train in   fp16


## 3. Cap this process to 4 GB

`set_per_process_memory_fraction` caps PyTorch's allocator. Everything after this cell
runs as if the card were a 4 GB laptop GPU — an allocation past the cap raises, exactly
as it would on the real thing.

**One historical caveat, now closed.** `torch.cuda.mem_get_info()` reports the whole
card, so Kadhi's own pre-flight check used to read that and print a "free VRAM" line that
belonged to the host card, not to this capped process — it would have allowed a
configuration that the allocator then refused. That was
#347, closed by
`training.stream_vram_override` in v0.73.1: section 5's config below sets it to this
notebook's own 4 GB cap, so the pre-flight checks against the same number the allocator
enforces.


In [5]:
BUDGET_BYTES = 4 * 1000**3  # 4 GB, the card this method was developed on

fraction = BUDGET_BYTES / total
torch.cuda.set_per_process_memory_fraction(fraction)
print(f"capped at {BUDGET_BYTES / 1e9:.2f} GB  (fraction {fraction:.3f} of this card)")

# Prove the cap bites: ask for 15% more than the budget and expect a refusal.
try:
    _ = torch.empty(int(BUDGET_BYTES * 1.15), dtype=torch.uint8, device="cuda")
    print("WARNING: the allocation succeeded — the cap is NOT in force")
except RuntimeError as exc:
    print("refused, as it should be:", str(exc).splitlines()[0][:90])


capped at 4.00 GB  (fraction 0.256 of this card)
refused, as it should be: CUDA out of memory. Tried to allocate 4.29 GiB. GPU 0 has a total capacity of 14.56 GiB of


## 4. The claim that matters: streamed == resident, bit for bit

A streaming bug is silent. If the base were substituted wrongly, or the autograd path
severed, the loss would still fall — the upper layers keep learning — and you would ship
a damaged model without an error anywhere.

So the check is not "does it train". It is: **the same weights, through the same kernels,
must produce the same numbers.** Below, one model is streamed layer-by-layer and the
other is an ordinary resident model, both carrying identical adapter weights.
`torch.equal` is exact equality, not a tolerance.

This runs on a small model because the reference has to fit in memory next to the
streamed copy — that is the whole reason the headline size cannot be checked this way.


In [6]:
import tempfile
from pathlib import Path

from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM

from kadhi_cli.utils.layer_shard import shard_checkpoint
from kadhi_cli.utils.layer_stream import resolve_stream_dtype
from kadhi_cli.utils.layer_stream_runtime import build_streamed_model
from kadhi_cli.utils.spectrum_scan import resolve_model_weights

MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
DTYPE = resolve_stream_dtype("cuda")  # fp16 on a T4, bf16 on an Ampere card
LORA = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.0, bias="none",
    target_modules=["q_proj", "v_proj"], task_type=TaskType.CAUSAL_LM,
)

workdir = Path(tempfile.mkdtemp())
weights = resolve_model_weights(MODEL)  # downloads on first use
index = shard_checkpoint(weights, str(workdir / "shards"), dtype=DTYPE, arch="llama")
streamed, runtime = build_streamed_model(
    model_id=weights, shard_dir=str(workdir / "shards"), index=index,
    lora_config=LORA, device="cuda", dtype=DTYPE, buffers=2, pin=True, seed=0,
)
print(f"streamed: {index.n_layers} layers, dtype={DTYPE}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


streamed: 30 layers, dtype=float16


In [7]:
# PEFT initialises lora_B to zero, so an untrained adapter contributes NOTHING and any
# comparison would silently be about the base model alone. Make it load-bearing first.
gen = torch.Generator().manual_seed(7)
with torch.no_grad():
    for pname, param in streamed.named_parameters():
        if "lora_B" in pname:
            param.copy_(torch.randn(param.shape, generator=gen).to(param.device, param.dtype))

resident = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=getattr(torch, DTYPE), device_map={"": "cuda"}
)
resident = get_peft_model(resident, LORA)

# Copy the adapter across. The streamed wrapper inserts an '.inner.' segment in its keys.
src = {k.replace(".inner.", "."): v for k, v in streamed.state_dict().items() if "lora_" in k}
dst = {k.replace(".inner.", "."): v for k, v in resident.state_dict().items() if "lora_" in k}
assert src and set(src) == set(dst)
with torch.no_grad():
    for key, val in src.items():
        dst[key].copy_(val.to(dst[key].dtype))

ids = torch.randint(0, 4096, (1, 32), device="cuda")
with torch.no_grad():
    a = streamed(input_ids=ids).logits
    b = resident(input_ids=ids).logits

print("max |streamed - resident| =", (a.float() - b.float()).abs().max().item())
print("torch.equal              =", torch.equal(a, b))
assert torch.equal(a, b), "NOT bit-exact — please open an issue with this output"
print("\nBit-exact. The streamed model is the same model.")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from '/root/.kadhi/spectrum/weights/HuggingFaceTB__SmolLM2-135M-Instruct' to 'HuggingFaceTB/SmolLM2-135M-Instruct'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(


max |streamed - resident| = 0.0
torch.equal              = True

Bit-exact. The streamed model is the same model.


In [8]:
import gc

# Free the reference before the headline run.
runtime.close()
del streamed, resident, a, b
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print("reset")


reset


## 5. The headline: Llama-3.1-8B, trained under the 4 GB cap

8B parameters. In NF4 the weights alone are about **4.5 GB** — more than the budget this
process is allowed, before activations, gradients or the optimizer. It trains anyway,
because at any moment only a couple of decoder layers are resident.

Expect ~20 minutes, most of it the download. The number to watch is the **peak VRAM**
at the end — not the loss. This is a handful of steps on 32 toy rows; over that distance
the loss can go up as easily as down, and it would prove nothing either way. What is
being demonstrated is that the run *happens at all* inside the budget.


In [9]:
import json

# Varied on both sides on purpose: 32 copies of one answer drive the loss to 0.000
# immediately and the printed curve stops meaning anything.
TOPICS = [
    ("streaming", "Only a couple of decoder layers are resident at any moment."),
    ("NF4", "Four-bit weights make the host-side store about four times smaller."),
    ("LoRA", "The base is frozen, so it is read and never written."),
    ("VRAM", "Peak memory is bounded by one layer instead of by the model."),
]
rows = [
    {"messages": [
        {"role": "user", "content": f"Question {i}: tell me about {topic}."},
        {"role": "assistant", "content": answer},
    ]}
    for i in range(8)
    for topic, answer in TOPICS
]
Path("train.jsonl").write_text("\n".join(json.dumps(r) for r in rows), encoding="utf-8")

config = """
base: NousResearch/Meta-Llama-3.1-8B-Instruct
task: sft
data:
  train: train.jsonl
  max_length: 256
training:
  epochs: 1
  batch_size: 1
  lr: 0.0002
  logging_steps: 1        # short run — without this nothing gets logged
  quantization: 4bit      # NF4 — the store is ~4x smaller than bf16
  stream_layers: true     # the feature
  stream_buffers: 2
  stream_vram_override: 4000000000  # same 4 GB the allocator enforces above (#347)
  lora:
    r: 8
    alpha: 16
output: ./out-8b
"""
Path("kadhi.yaml").write_text(config, encoding="utf-8")
print(config)



base: NousResearch/Meta-Llama-3.1-8B-Instruct
task: sft
data:
  train: train.jsonl
  max_length: 256
training:
  epochs: 1
  batch_size: 1
  lr: 0.0002
  logging_steps: 1        # short run — without this nothing gets logged
  quantization: 4bit      # NF4 — the store is ~4x smaller than bf16
  stream_layers: true     # the feature
  stream_buffers: 2
  stream_vram_override: 4000000000  # same 4 GB the allocator enforces above (#347)
  lora:
    r: 8
    alpha: 16
output: ./out-8b



The cell below runs the trainer **in this process** rather than shelling out to
`kadhi train`. That is deliberate and it is the only reason: `max_memory_allocated()`
reports the peak of *the process that calls it*, so a subprocess would train fine and
leave us measuring nothing. It is the same code path the CLI runs on the same
`kadhi.yaml` above — the CLI adds argument parsing and the pre-flight panel, neither of
which changes what the GPU does.


In [10]:
from kadhi_cli.config.loader import load_config_from_string
from kadhi_cli.data.loader import load_dataset
from kadhi_cli.trainer.sft import SFTTrainerWrapper

cfg = load_config_from_string(config)
dataset = load_dataset(cfg.data)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

wrapper = SFTTrainerWrapper(cfg)
wrapper.setup(dataset)   # downloads, shards to NF4, builds the streamed model
result = wrapper.train()

print(f"\nsteps: {result['total_steps']}  loss: {result['initial_loss']:.3f}"
      f" -> {result['final_loss']:.3f}")


Auto-detected format: chatml

Loading tokenizer: NousResearch/Meta-Llama-3.1-8B-Instruct

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

Truncating train dataset:   0%|          | 0/28 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
1,4.529099
2,4.441022
3,3.946751
4,4.775282
5,4.251106
6,4.155877
7,4.029544



steps: 7  loss: 4.529 -> 4.030


In [11]:
peak = torch.cuda.max_memory_allocated()
print(f"peak VRAM allocated by this process: {peak / 1e9:.2f} GB")
print(f"budget this process was capped to:   {BUDGET_BYTES / 1e9:.2f} GB")
print("model weights in NF4, for scale:     ~4.5 GB")

adapter = Path("out-8b/adapter_model.safetensors")
print(f"\nadapter written: {adapter.exists()}")
if adapter.exists():
    from safetensors.torch import load_file

    tensors = load_file(str(adapter))
    live = sum(1 for v in tensors.values() if v.abs().max().item() > 0)
    print(f"adapter tensors: {len(tensors)}, non-zero: {live}")


peak VRAM allocated by this process: 1.83 GB
budget this process was capped to:   4.00 GB
model weights in NF4, for scale:     ~4.5 GB

adapter written: True
adapter tensors: 128, non-zero: 128


## What this proved, and what it did not

**Proved, on your hardware:**

- A streamed model returns **bit-identical** logits to an ordinary one (§4).
- An 8B model trained with a measured peak below a cap smaller than its own weights (§5).
- Both on a GPU with **no bf16** — the case that was broken until recently.

**Not proved:**

- *Backward* exactness at this size. §4 compares the forward. Gradient exactness is
  verified up to 14B against resident references on hardware that can hold them, and
  a defect **above** that size was found, named upstream and repaired — see
  `benchmarks/`.
- Speed. A T4 under an artificial cap is not a throughput benchmark, and this notebook
  deliberately does not quote tok/s.

Layer streaming is **BETA** and opt-in (`stream_layers: true`).

**If any assertion above failed, that is worth an issue** — with the cell output. A
reproduction on hardware we do not own is more useful to this project than a star.

The method, the correctness protocol and every measurement:
[10.5281/zenodo.21771064](https://doi.org/10.5281/zenodo.21771064) ·
measurement records
